# Addestramento di raid-projexa-14b (Projexa, Qwen3 14B)

Addestra **raid-projexa-14b**, il modello che ricava dal recap di una riunione i **rischi, le decisioni, le dipendenze e le change request** quando l'AI è **Projexa 14B** (in automatico con AI Automatizza dopo il recap, o dal pulsante).
Gli esempi sono le scelte della pagina **Analisi PM › RAID e CR**: per ogni estrazione la risposta giusta tra quella di Projexa e quella dell'AI esterna.
Il risultato è un adattatore (`raid-projexa-14b-adapter.gguf`, 200-500 MB) da caricare in **Analisi PM › Modello sulla VM**, scegliendo **raid-projexa-14b**.

- Sulla GPU gratuita T4 si addestra con testi fino a **6.144 token**: i recap molto lunghi si saltano da soli. Se la cella 4 va in errore di memoria (*CUDA out of memory*), alla cella 3 metti `LUNGHEZZA = 4096` e rifai da lì.
- Il ragionamento ad alta voce dei Qwen3 è spento (come sulla VM).
- Tempo indicativo: 30-90 minuti.

**Prima di iniziare**: menu **Runtime → Cambia tipo di runtime → GPU T4** → Salva. Poi **Runtime → Esegui tutto** e, alla cella 2, carica il file .jsonl.
Se una cella dà errore, copia il messaggio e mandalo a chi ti segue lo sviluppo: le librerie cambiano spesso versione (notebook non ancora provato).

## 1. Installazione delle librerie (5-10 minuti)

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), 'Nessuna GPU: Runtime > Cambia tipo di runtime > GPU T4'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Carica il file degli esempi (.jsonl scaricato da Projexa)

In [ ]:
from google.colab import files
import json, collections
caricati = files.upload()
nome = next(n for n in caricati if n.endswith('.jsonl'))
esempi = [json.loads(r) for r in open(nome, encoding='utf-8') if r.strip()]
def tipo(e):
    r = json.loads(e['messages'][-1]['content'])
    return 'con voci' if any(r.get(k) for k in ('rischi', 'decisioni', 'dipendenze', 'change_request')) else 'niente da registrare'
print(len(esempi), 'esempi:', dict(collections.Counter(tipo(e) for e in esempi)))

## 3. Modello base (Qwen3 14B, lo stesso usato da RAID e CR sulla VM)

In [ ]:
from unsloth import FastLanguageModel
LUNGHEZZA = 6144
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3-14B-unsloth-bnb-4bit', max_seq_length=LUNGHEZZA, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)


## 4. Addestramento

Il testo di ogni esempio si prepara con il ragionamento spento (`enable_thinking=False`), come lo usa Projexa.


In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
def testo(e):
    try:
        return tokenizer.apply_chat_template(e['messages'], tokenize=False, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(e['messages'], tokenize=False)
testi = [testo(e) for e in esempi]
tenuti = [t for t in testi if len(tokenizer(t)['input_ids']) <= LUNGHEZZA]
print(f'{len(tenuti)} esempi usati, {len(testi) - len(tenuti)} saltati perché troppo lunghi per la T4')
assert tenuti, 'Nessun esempio abbastanza corto: servono recap più brevi'
dati = Dataset.from_dict({'text': tenuti}).shuffle(seed=42)
epoche = 3 if len(tenuti) < 150 else 2
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dati,
    args=SFTConfig(dataset_text_field='text', max_seq_length=LUNGHEZZA, per_device_train_batch_size=1,
                   gradient_accumulation_steps=8, num_train_epochs=epoche, learning_rate=1e-4,
                   warmup_steps=5, logging_steps=5, optim='adamw_8bit', fp16=True, bf16=False,
                   output_dir='uscita', report_to='none', seed=42))
trainer.train()


## 5. Prova veloce (un recap breve)

In [ ]:
FastLanguageModel.for_inference(model)
compito = esempi[0]['messages'][0]['content']
inizio = compito[:compito.rindex('TESTO DELLA RIUNIONE')]
recap = 'Il cliente chiede un report in più sulle presenze, da avere entro inizio dicembre. Si decide di rilasciarlo con un hotfix sulla 5.27. Lo sviluppo dipende dalla disponibilità del team prodotto.'
prova = [{'role': 'user', 'content': inizio + 'TESTO DELLA RIUNIONE «Prova» del 22/10/2026\n' + recap}]
try:
    ids = tokenizer.apply_chat_template(prova, add_generation_prompt=True, return_tensors='pt', enable_thinking=False).to('cuda')
except TypeError:
    ids = tokenizer.apply_chat_template(prova, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=600, temperature=0.1)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 6. Adattatore in formato Ollama (.gguf) e download

In [ ]:
model.save_pretrained('adattatore')
tokenizer.save_pretrained('adattatore')
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python llama.cpp/convert_lora_to_gguf.py adattatore --base-model-id Qwen/Qwen3-14B --outtype f16 --outfile raid-projexa-14b-adapter.gguf
import os
print('Dimensione:', round(os.path.getsize('raid-projexa-14b-adapter.gguf') / 1e6), 'MB')
files.download('raid-projexa-14b-adapter.gguf')


## 7. Fatto

Carica `raid-projexa-14b-adapter.gguf` in Projexa: **Analisi PM → Modello sulla VM**, scegli **raid-projexa-14b** e premi **Carica e installa** (non durante una call con l'Assistente PM).
Projexa fa il backup del modello attuale, installa il nuovo sopra la base qwen3:14b e lo prova; da quel momento RAID e Change Request con «Projexa 14B» usano il modello addestrato.